# 06 — Urban Canopy Parameters (UCP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/06_urban_canopy_parameters.en.ipynb)

**Learning objective.** By the end of this notebook you will use `lcz_get_ucp` to download and process
Urban Canopy Parameters — built-up density/height/volume (GHSL), building morphology and land-fraction
metrics (WUDAPT/WUMPOD), vegetation/impervious cover, and directional street-canyon roughness — either
as a gridded raster stack over a city or extracted at specific point locations (e.g. weather stations),
and you will understand why this station-level extraction is the missing link between remote-sensing
urban form and station-based urban climate analysis.

## From LCZ class to physical urban form: what UCPs add

The LCZ scheme (Stewart & Oke, 2012) is deliberately compact: it sorts every urban and natural surface
into one of 17 discrete classes on the basis of a handful of *typical* morphological and land-cover
properties (building height, building/tree spacing, imperviousness, sky view factor). That
discretization is what makes LCZ so useful as a common reporting unit across cities — but it also
throws away information. Two locations both mapped as "LCZ 2 — compact midrise" can have quite
different actual building heights, impervious fractions, or vegetation cover; the LCZ class only tells
you the *typical range*, not the *pixel-level* value at a given spot.

**Urban Canopy Parameters (UCPs)** are the continuous, physically-based variables that fill this gap.
Where LCZ4r's `lcz_get_parameters` (notebook 03) derives 34 morphological parameters analytically from
the LCZ class itself (e.g. "LCZ 2 has a typical building surface fraction of 0.4-0.6"), `lcz_get_ucp`
goes a level deeper and pulls **directly observed, remote-sensing-derived** urban form data for the
exact footprint of your study area, independent of which LCZ class each pixel happens to fall into.
Four families of UCPs are processed:

- **Built-up density/height/volume (GHSL)** — the Global Human Settlement Layer (Melchiorri et al.,
  2025, JRC) gives built-up *surface fraction* (m²/m², how much of a pixel is covered by buildings),
  *building height* (m, above ground), *built-up volume* (m³, height × footprint — a proxy for total
  building mass/thermal capacity), and *population density* (persons/km²). These four together capture
  the "how much city is here, and how tall/dense is it" question that drives both the urban heat
  storage capacity and the anthropogenic heat load of a neighborhood.
- **WUDAPT/WUMPOD morphology (Patel & Roth, 2022/2024)** — building fraction, height, length,
  perimeter, land fraction, plus land-cover classifications (CGLC, global urban fraction). These
  supply the geometric inputs (`plan area index`, `building surface-to-plan ratio`) that classic urban
  canopy and single-layer urban energy balance models require.
- **Vegetation/impervious cover (GLC_FCS30D, Zhang et al., 2021)** — tree cover % and impervious
  surface % at 30 m, a direct measurement of the vegetation-fraction/impervious-fraction balance that
  is the single strongest physical driver of daytime Urban Heat Island intensity.
- **Directional (street-canyon) roughness** — land fraction, height index, and aerodynamic roughness
  parameters (zero-plane displacement `zdm`, roughness length `zdr`/`zom`/`zor`) computed along four
  compass directions (0°, 45°, 90°, 135°). These describe how building geometry varies with wind
  approach direction — critical for modeling street-canyon ventilation, pollutant dispersion, and the
  aerodynamic drag a city imposes on the boundary-layer wind field.

**Why extract UCPs *at station locations*, specifically?** A weather station's raw temperature reading
is the product of the atmosphere *and* whatever is immediately around the sensor: a station sitting in
a small park surrounded by tall buildings will read differently than one in an open, low-rise
residential street, even at identical regional forcing. If you only know a station's LCZ class, you
know its *category* of surroundings; if you additionally extract its building height, impervious
fraction, and canopy density (which is exactly what `lcz_get_ucp(..., stations=...)` does), you can
build a *quantitative, physically interpretable* model of why station A is consistently 2°C warmer than
station B. This is precisely the feature set that the machine-learning spatial interpolation notebook
(`notebooks/local/05_ml_interpolation_ucp`) trains a Random Forest on to predict temperature
pixel-by-pixel across a whole city grid, using UCPs as predictors instead of (or alongside) simple
spatial coordinates. `lcz_get_ucp` is the data-acquisition step that feeds that model.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Get the LCZ map

As in every notebook in this series, we start from `lcz_get_map`, which geocodes a city name, downloads
only the pixels covering it from the global LCZ Zenodo Cloud-Optimized GeoTIFF, and returns a local
path. We use Juiz de Fora again — its tiny footprint keeps every download in this notebook fast.

In [2]:
from LCZ4py import lcz_get_map

lcz_map_path = lcz_get_map(city="Juiz de Fora")
print(lcz_map_path)

13:37:34 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


13:37:38 - rasterio._env - WARNING - CPLE_IllegalArg in tmpp0wvsx9t.tif: BLOCKXSIZE can only be used with TILED=YES


13:37:38 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


/Users/co2map/.lcz4r_cache/clipped_c7031cb44f08aaf6.tif


## 2. `lcz_get_ucp`: signature and what you get back

```python
def lcz_get_ucp(
    lcz_map,                      # path/DataArray defining the study-area footprint
    stations=None,                # None -> raster stack; GeoDataFrame/DataFrame/path -> point extraction
    cache_dir="lcz4r_cache",      # disk cache for downloaded/processed rasters
    n_workers=None,                # parallel download workers (default: CPU count - 1)
    verbose=True,
    variables=None,                # subset of variable names, e.g. ["elevation", "tree", "lf_0"]
    ghsl_tiles=None,                # specific GHSL tile IDs (None = auto-detect from lcz_map extent)
    process_ghsl=True,             # built-up surface/height/volume + population (GHSL)
    process_wumpod=True,           # building morphology + land-cover (WUDAPT/WUMPOD)
    process_vegetation=True,       # tree cover / impervious surface (GLC_FCS30D)
    process_directional=True,      # street-canyon roughness at 0/45/90/135 degrees
    use_threads=True,              # must stay True (see below)
    fail_fast=False,                # if True, raise on first failed variable instead of skipping it
) -> dict
```

Internally this wraps an `UrbanParameterProcessor` that downloads every requested variable **in
parallel** via GDAL's `/vsicurl/` streaming (so, like the other `lcz_get_*` functions in this package,
only the pixels intersecting your `lcz_map` footprint are ever transferred, not full global rasters),
resamples everything onto a common grid, and either (a) extracts values at your station coordinates or
(b) mosaics the result into a raster stack. `use_threads=False` deliberately raises — the processor
holds a `requests.Session` and a `diskcache.FanoutCache` on `self`, neither of which survives being
pickled to a separate process, and since these are I/O-bound downloads, threads already parallelize
them effectively.

**Return value** — a plain `dict` (not a dataclass, unlike most other `lcz_get_*` functions), because
its shape depends on whether `stations` was given:

- `'df_vars'` — a `pandas.DataFrame` with one row per station and one column per successfully processed
  variable, plus the identifier column (`None` if `stations=None`).
- `'combined_rasters'` — an `xarray.Dataset` holding every processed variable as its own 2-D
  `DataArray`, already resampled/aligned to `lcz_map`'s grid.
- `'stack_path'` — path to the saved GeoTIFF stack `LCZ4r_output/lcz_ucp_stack.tif` (only when
  `stations=None`; otherwise `None`, since in that case the per-station table is the deliverable).
- `'variable_list'` — list of variable names that downloaded and processed successfully.
- `'failed_variables'` — list of `(variable_name, error_message)` tuples for anything that failed
  (network hiccups, missing tiles, etc. — the processor is fault-tolerant by default: `fail_fast=False`
  means one failed variable doesn't abort the whole run).
- `'summary'` — a `ProcessingSummary` dataclass: `total_variables`, `successful`, `failed`,
  `total_time` (seconds), `failed_variables`.

We will exercise both branches below: first `stations=None` (raster-stack mode), then `stations=<a few
points>` (station-extraction mode, the one that matters for the local/05 ML-interpolation workflow).

### 2a. Raster-stack mode (`stations=None`)

For a first pass we restrict `variables` to a handful of cheap, illustrative parameters from each of
the non-GHSL families — `elevation`, `tree` (vegetation), `urban` (impervious surface), and three
directional roughness metrics (`lf_0`, `hi_45`, `zdm_90`) — and turn off `process_ghsl` and the full
`process_directional` sweep. This keeps the download tiny (a handful of small cropped rasters instead
of dozens) while still showing every UCP family except GHSL in action. GHSL is treated separately in
section 4 below because — unlike the WUDAPT/WUMPOD and vegetation sources, which are served as plain
Cloud-Optimized GeoTIFFs — GHSL tiles are distributed as zipped archives, so `/vsicurl/` has to stream
through `/vsizip/` on top, which is considerably slower even for a single small tile (each GHSL variable
took 30-125 seconds for Juiz de Fora's single tile in testing, vs. a few seconds for the WUMPOD/vegetation/
directional variables below).

In [3]:
from LCZ4py.general.lcz_get_ucp import lcz_get_ucp

result_stack = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=None,
    n_workers=4,
    variables=["elevation", "tree", "urban", "lf_0", "hi_45", "zdm_90"],
    process_ghsl=False,
    process_wumpod=True,
    process_vegetation=True,
    process_directional=True,
)

print(result_stack["variable_list"])
print(result_stack["failed_variables"])
print(result_stack["summary"])
print(result_stack["stack_path"])
result_stack["combined_rasters"]

13:37:38 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:37:38 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


13:37:38 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:37:38 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: -43.686_-22.0_-43.146_-21.52


13:37:38 - LCZ4py.general.lcz_get_ucp - INFO - Target CRS: EPSG:4326


...

13:37:39 - LCZ4py.general.lcz_get_ucp - INFO - Raster layers: 6


13:37:39 - LCZ4py.general.lcz_get_ucp - INFO - Stack saved: LCZ4r_output/lcz_ucp_stack.tif


13:37:39 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


['elevation', 'lf_0', 'hi_45', 'zdm_90', 'urban', 'tree']
[]
ProcessingSummary(total_variables=6, successful=6, failed=0, total_time=0.449260950088501, failed_variables=[])
LCZ4r_output/lcz_ucp_stack.tif


<xarray.Dataset> Size: 13MB
Dimensions:      (x: 601, y: 534)
Coordinates:
  * x            (x) float64 5kB -43.69 -43.68 -43.68 ... -43.15 -43.15 -43.15
  * y            (y) float64 4kB -21.52 -21.52 -21.52 ... -22.0 -22.0 -22.0
    spatial_ref  int64 8B 0
Data variables:
    elevation    (y, x) float32 1MB nan nan nan nan nan ... nan nan nan nan nan
    lf_0         (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    hi_45        (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    zdm_90       (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    urban        (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    tree         (y, x) float32 1MB nan nan nan nan nan ... nan nan nan nan nan

`result_stack['summary']` reports how many of the requested variables succeeded/failed and the total
wall-clock time; with the network cache warm (as it will be after your first run of this notebook) this
finishes in a few seconds. `result_stack['combined_rasters']` is an `xarray.Dataset` — one `DataArray`
per variable, already cropped and resampled to the same grid as `lcz_map_path` — and `stack_path` is
where that same data was written to disk as a multi-band GeoTIFF (`lcz_ucp_stack.tif`, one band per
variable, band descriptions set to the variable name) since no stations were supplied. This raster
stack is exactly the format `lcz_plot_parameters` and `lcz_interp_map_plus`/`lcz_interp_eval_plus`
(notebook `local/05_ml_interpolation_ucp`) expect as a source of gridded predictor variables.

### 3. Visualizing the UCP raster stack

`lcz_plot_parameters` accepts the `dict` returned by `lcz_get_ucp(..., stations=None)` directly — no
need to reload it from disk. Passing `all_params=True` plots every variable in `combined_rasters` as its
own interactive map.

In [4]:
from LCZ4py.general.lcz_plot_parameters import lcz_plot_parameters

figs = lcz_plot_parameters(result_stack, all_params=True)
print(type(figs), len(figs))
figs[1]

<class 'list'> 6


Each panel is a continuous raster over Juiz de Fora's footprint — for instance, `tree` (tree cover %) should
be visibly higher over the forested hillslopes and lower in the built-up valley floor, while `urban`
(impervious surface %) shows the opposite pattern. Because these are pixel-level, directly observed
values (not values inferred from an LCZ class lookup table), you will see real within-class
heterogeneity here that `lcz_get_parameters`' analytic parameters cannot show — two pixels in the same
LCZ class can, and often do, have different tree cover.

### 4. Station-extraction mode (`stations=...`)

Now the mode that matters most for connecting UCPs to station-based climate analysis: pass a
`GeoDataFrame` of point locations and `lcz_get_ucp` extracts the value of every processed variable
at each point instead of returning a raster. In a real workflow these would be your weather-station
coordinates (e.g. the `Latitude`/`Longitude` columns of `lcz_data_berlin.csv`, used throughout the
`local/` notebook series); here we use three illustrative points scattered across Juiz de Fora's small
footprint so the example stays fast and self-contained.

In [5]:
import geopandas as gpd
from shapely.geometry import Point

stations = gpd.GeoDataFrame(
    {"station": ["S1", "S2", "S3"]},
    geometry=[Point(9.52, 47.13), Point(9.55, 47.14), Point(9.58, 47.15)],
    crs="EPSG:4326",
)

result_stations = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=stations,
    n_workers=4,
    variables=["elevation", "tree", "urban"],
    process_ghsl=False,
    process_wumpod=True,
    process_vegetation=True,
    process_directional=False,
)

result_stations["df_vars"]

13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: -43.686_-22.0_-43.146_-21.52


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - Target CRS: EPSG:4326


...

13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - ⏱️  Total Time: 0.3s


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - Raster layers: 3


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - DataFrame: 3 rows × 4 cols


13:37:40 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


,station,elevation,tree,urban
0,S1,NaN,NaN,NaN
1,S2,NaN,NaN,NaN
2,S3,NaN,NaN,NaN


`df_vars` is now a per-station table: one row per station, one column per successfully processed
variable, joined on the station identifier column. This is the table you would feed straight into
`lcz_interp_map_plus`/`lcz_interp_eval_plus` (`ucp_variables=...`) as the feature matrix for a
Random-Forest temperature-interpolation model, or merge with your air-temperature observations to
regress "how much of station A vs. station B's temperature difference is explained by tree cover /
impervious fraction / building height" — turning a purely categorical LCZ-based explanation into a
continuous, quantitative one. `stack_path` is `None` here (as documented above) since the raster stack
is not the deliverable when stations are supplied.

## 5. GHSL built-up density/height/volume/population (reference — not executed here)

`process_ghsl=True` additionally downloads and mosaics four GHSL layers — `built_surface`,
`built_height`, `built_volume`, `pop` — auto-detecting which of GHSL's global tile grid cells intersect
your `lcz_map` footprint (or accepting an explicit `ghsl_tiles=[...]` list). The code below is exactly
what you would run:

```python
result_ghsl = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=None,
    n_workers=4,
    process_ghsl=True,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=False,
)
print(result_ghsl["variable_list"], result_ghsl["failed_variables"])
```

**This cell is deliberately left un-executed.** In testing, even for Juiz de Fora's single tiny GHSL tile,
each of the four variables took 30 seconds to over 2 minutes to stream (GHSL tiles are served as zipped
archives, so GDAL has to open them via a `/vsizip//vsicurl/` chain rather than plain `/vsicurl/`, which
is markedly slower than the WUDAPT/WUMPOD/vegetation/directional sources above) — roughly 4-5 minutes
total for four variables and one tile. For a larger city spanning multiple GHSL tiles, or when several
notebooks/agents share the same network path, this easily exceeds what is reasonable to bake into a
tutorial's automated execution. If you run this yourself, expect it to take several minutes even for a
small city, and budget proportionally more for a metropolitan-scale one; the payoff is the same
zero-to-population-density built-environment layers GHSL is widely used for in urban climate and
exposure studies.

## 6. Directional street-canyon roughness, full sweep (reference)

Section 2a already exercised three directional variables (`lf_0`, `hi_45`, `zdm_90`) to prove the
mechanism works and is fast. The *full* `process_directional=True` sweep (the package default) computes
land fraction and height index at each of four directions, plus four aerodynamic roughness parameters
(`zdm`, `zdr`, `zom`, `zor`) at each direction — around 30 variables in total:

```python
result_directional = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=None,
    n_workers=8,
    process_ghsl=False,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=True,   # ~30 variables: lf_*, hi_*, zdm_*, zdr_*, zom_*, zor_* at 0/45/90/135°
)
```

Each of these variables downloads about as fast as the three we already ran (a few seconds, since
they come from the same plain-GeoTIFF Zenodo source as WUMPOD, not the slower zipped GHSL archives), so
the full sweep mainly costs more *variables*, not more *time per variable* — with `n_workers` raised to
match, it is a reasonable one to run yourself for a full urban-morphology characterization, but we keep
the executed demo above to the three-variable subset to keep this notebook's total runtime short.

## Conclusion & next steps

`lcz_get_ucp` bridges remote-sensing-derived urban form (GHSL built-up density/height/volume/
population, WUDAPT/WUMPOD building morphology, GLC_FCS30D vegetation/impervious cover, and directional
street-canyon roughness) with two consumption modes: a gridded raster stack for city-wide mapping and
visualization, or point extraction at specific coordinates — most importantly, weather-station
locations — for attributing station-level temperature differences to the physical urban form actually
surrounding each sensor. That station-level UCP table is precisely the feature set the ML-based spatial
interpolation notebook trains on.

**Previous:** [`05_spectral_indices`](./05_spectral_indices.en.ipynb) — NDVI/NDBI/MNDWI/EVI per LCZ
class from Sentinel-2.

**Next:** [`07_gridded_climate_environment`](./07_gridded_climate_environment.en.ipynb) — downloading
gridded climate (ERA5, CHIRPS) and environmental (pollution, drought) variables cropped to an LCZ map,
the last notebook in the `general/` series before the station-based `local/` series
([`local/01_lcz_time_series`](../local/01_lcz_time_series.en.ipynb)), which is where the UCP-at-stations
table produced here feeds directly into
[`local/05_ml_interpolation_ucp`](../local/05_ml_interpolation_ucp.en.ipynb).